## Chapter 5 Exercises from Del Prado Textbook

In [2]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import plotly.graph_objects as go
import plotly.io as pio
from statsmodels.tsa.stattools import adfuller
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

In [19]:
# Functions for the later exercises

def run_adf(series, name="series"):
    series = pd.Series(series).dropna()
    result = adfuller(series, maxlag=1, regression="c", autolag=None)

    print(f"\n{name}")
    print(f"ADF statistic: {result[0]:.4f}")
    print(f"p-value:       {result[1]:.4f}")
    return result[0], result[1]


def sine_func(t, A, omega, phi, C):
    return A * np.sin(omega * t + phi) + C


def fit_sine_r2(y):
    y = np.asarray(y)
    t = np.arange(len(y))

    # Initial guesses
    A0 = (np.max(y) - np.min(y)) / 2
    omega0 = 2 * np.pi / 50
    phi0 = 0
    C0 = np.mean(y)

    try:
        params, _ = curve_fit(sine_func, t, y, p0=[A0, omega0, phi0, C0], maxfev=10000)
        y_hat = sine_func(t, *params)
        return r2_score(y, y_hat), params
    except RuntimeError:
        return np.nan, None

def get_fracdiff_weights(d, size, thresh=1e-5):
    """
    Compute fractional differencing weights.
    """
    w = [1.0]

    for k in range(1, size):
        w_k = -w[-1] * (d - k + 1) / k

        if abs(w_k) < thresh:
            break

        w.append(w_k)

    return np.array(w[::-1]).reshape(-1, 1)


def frac_diff(series, d, thresh=1e-5):
    """
    Fixed-width fractional differencing.
    """
    series = pd.Series(series).dropna()
    weights = get_fracdiff_weights(d, len(series), thresh)

    width = len(weights) - 1
    output = pd.Series(index=series.index, dtype=float)

    for i in range(width, len(series)):
        window = series.iloc[i - width:i + 1]
        output.iloc[i] = np.dot(weights.T, window)[0]

    return output.dropna()

def find_min_d_for_stationarity(series, tau=1e-2, alpha=0.05):
    d_values = np.linspace(0, 1, 101)

    results = []

    for d in d_values:
        fd_series = frac_diff(series, d, thresh=tau)

        if len(fd_series) < 20:
            continue

        adf_stat, p_val = adfuller(fd_series, maxlag=1, regression="c", autolag=None)[:2]

        results.append((d, adf_stat, p_val))

        if p_val < alpha:
            return d, adf_stat, p_val, pd.DataFrame(results, columns=["d", "ADF", "p_value"])

    return None, None, None, pd.DataFrame(results, columns=["d", "ADF", "p_value"])

### 5.1

In [4]:
# (a)
np.random.seed(42) # for repro ducibility

n = 1000
x = pd.Series(np.random.normal(0, 1, n))
adf_x, p_x = run_adf(x, "IID Gaussian series")


IID Gaussian series
ADF statistic: -22.3925
p-value:       0.0000


In [11]:
# (b)

y = x.cumsum()
# The cumulative series is integrated of order 1, so it is I(1).

# (ii)
adf_y, p_y = run_adf(y, "Cumulative sum series")
# As expected, the cumulative sum is non-stationary, so the ADF p-value should be large.


Cumulative sum series
ADF statistic: -0.9504
p-value:       0.7709


In [12]:
# (c)
x_diff2 = x.diff().diff().dropna()
adf_x_diff2, p_x_diff2 = run_adf(x_diff2, "Twice-differenced IID Gaussian series")


Twice-differenced IID Gaussian series
ADF statistic: -51.6798
p-value:       0.0000


### 5.2

In [15]:
# (a)
n = 1000
t = np.arange(n)
sin_series = pd.Series(np.sin(2 * np.pi * t / 50))

adf_sin, p_sin = run_adf(sin_series, "Sine series")



Sine series
ADF statistic: -40703803017929.2031
p-value:       0.0000


In [21]:
# (b)
shifted_sin = sin_series + 1
cum_shifted_sin = shifted_sin.cumsum()

adf_cum_sin, p_cum_sin = run_adf(cum_shifted_sin, "Cumulative shifted sine series")

# (ii)
d_tau_1e2, adf_tau_1e2, p_tau_1e2, results_tau_1e2 = find_min_d_for_stationarity(cum_shifted_sin, tau=1e-2, alpha=0.05)

print("="*80)
print("tau = 1e-2")
print("minimum d:", d_tau_1e2)
print("ADF:", adf_tau_1e2)
print("p-value:", p_tau_1e2)


Cumulative shifted sine series
ADF statistic: -0.6919
p-value:       0.8488
tau = 1e-2
minimum d: 0.97
ADF: -3.9016234858372565
p-value: 0.002022931578396388


### 5.3 

In [23]:
# (a)
r2_original, params_original = fit_sine_r2(cum_shifted_sin)

print("Original cumulative shifted sine series")
print("R-squared:", r2_original)
print("Params:", params_original)

Original cumulative shifted sine series
R-squared: 0.0019023493824669169
Params: [-1.78076597e+01  1.25620454e-01  5.48076120e-01  5.08444529e+02]


In [24]:
# (b)

series_ffd_1 = frac_diff(series_2b, d=1, thresh=1e-5)
r2_ffd_1, params_ffd_1 = fit_sine_r2(series_ffd_1)

print("FFD(d=1)")
print("R-squared:", r2_ffd_1)
print("Params:", params_ffd_1)

FFD(d=1)
R-squared: 1.0
Params: [1.         0.12566371 0.12566371 1.        ]


In [25]:
# (c) 

r2_results = []

for d in np.linspace(0, 1, 101):
    fd_series = frac_diff(series_2b, d=d, thresh=1e-5)

    if len(fd_series) < 20:
        continue

    r2, params = fit_sine_r2(fd_series)
    r2_results.append((d, r2))

r2_results = pd.DataFrame(r2_results, columns=["d", "R2"])

best_row = r2_results.loc[r2_results["R2"].idxmax()]

print("Best d:", best_row["d"])
print("Best R-squared:", best_row["R2"])

'''
The value of d that maximizes the sinusoidal R-squared should be close to 1.
This is because the series from 5.2(b) is the cumulative sum of a shifted sinusoid, so it is approximately integrated of order 1. 
Applying FFD with d near 1 removes the accumulated trend while preserving the original sinusoidal memory.
'''

Best d: 1.0
Best R-squared: 1.0
